# 05 — Translation + Emotion Tagging

**Purpose:** Translate each segment from source language to English, fitting the original speaker duration, with inline ElevenLabs v3 audio emotion tags.

## Why this is the hardest step
Straight translation ignores three constraints that are mandatory for dubbing:
1. **Duration fit** — the translated line must take the same time to say as the original. Over-translate and the TTS clip overlaps the next speaker. Under-translate and there is awkward silence.
2. **Emotional authenticity** — ElevenLabs v3 TTS is highly sensitive to audio tags. Without them, all output sounds neutrally polished, regardless of whether the original speaker was shouting or whispering.
3. **Lip-sync idiomatics** — English phrasing must match approximate mouth-shape timing of the original, favouring open vowels at key points.

## Architecture
1. Extract per-segment audio features (pitch, energy, speech rate) via librosa
2. Feed source text + audio features + duration target to Gemini
3. Verify duration fit (word count × per-emotion WPM estimate)
4. If outside ±15 %, **re-prompt** with explicit word budget and "previous was too long/short" — up to 3 real retries
5. Self-consistency: generate K=3 candidates, pick the one closest to target duration

## Metric rationale
- **Duration ratio** = estimated_spoken_duration / target_duration. Target: 0.85–1.15.
- **BLEU / chrF** vs. a reference-free baseline (simple Google Translate fallback) to detect catastrophic mistranslations.
- **Tag appropriateness** (subjective): are emotion tags consistent with source audio features?

## Per-emotion WPS adjustment
We adjust the target word count based on the detected emotion. Shouted lines are faster, whispered lines are slower. A global WPM ignores this and systematically over- or under-generates.

**API key needed:** `GEMINI_API_KEY`

**Input:** `transcription/transcription.json` + `stems/vocals.wav`  
**Output:** `translation/translation.json`


In [ ]:
!pip install -q google-generativeai librosa soundfile sacrebleu pandas tqdm
print('Ready.')


In [ ]:
import sys, os, json, time, re
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
from getpass import getpass
import numpy as np
import librosa
import pandas as pd
from tqdm.notebook import tqdm
import google.generativeai as genai

sys.path.insert(0, os.path.abspath('..'))
from config import (
    TRANSCRIPTION_DIR, TRANSLATION_DIR, STEMS_DIR,
    LANGUAGE_DISPLAY, SOURCE_LANGUAGE, TARGET_LANGUAGE,
    DURATION_TOLERANCE, MAX_DURATION_RETRIES, ALLOWED_AUDIO_TAGS,
    GEMINI_PRO_MODEL, GEMINI_FLASH_MODEL,
    ENGLISH_WPM_BY_EMOTION, ENGLISH_WPM_DEFAULT,
    SHOW_SUMMARY, SHOW_SETTING, SHOW_GENRES, LINGUISTIC_PROFILE, STYLE_TEMPLATE,
)

TRANSCRIPT_JSON = os.path.join(TRANSCRIPTION_DIR, 'transcription.json')
VOCALS_WAV      = os.path.join(STEMS_DIR, 'vocals.wav')
OUT_JSON        = os.path.join(TRANSLATION_DIR, 'translation.json')

with open(TRANSCRIPT_JSON, encoding='utf-8') as f:
    segments = json.load(f)
print(f'Loaded {len(segments)} ASR segments')
print(segments[0])


In [ ]:
# ── Show-level context — fill these in before running translation ──────────────
# These fill the system prompt placeholders for genre/world consistency.
SHOW_SUMMARY       = SHOW_SUMMARY       or 'A Tamil-language film or series (details pending).'
SHOW_SETTING       = SHOW_SETTING       or 'Contemporary India'
SHOW_GENRES        = SHOW_GENRES        or 'Drama'
LINGUISTIC_PROFILE = LINGUISTIC_PROFILE or 'Emotionally expressive conversational Tamil'
STYLE_TEMPLATE     = STYLE_TEMPLATE     or ''

SRC  = LANGUAGE_DISPLAY[SOURCE_LANGUAGE]
TGT  = LANGUAGE_DISPLAY[TARGET_LANGUAGE]
print(f'{SRC} -> {TGT}')


In [ ]:
# ── System prompt (Postudio template + dubbing-specific extensions) ────────────
# The base template is from translation_prompt (project root).
# Extensions add: ElevenLabs tags, duration constraints, audio-feature hints.
# {{}} in the template are literal braces (Python .format() escape).

SYSTEM_PROMPT_TEMPLATE = '''You are a professional dubbing translator and linguistic stylist specializing in adaptive translations for multilingual film and television.
Your objective is to produce dubbing-ready translations that feel native, emotionally authentic, genre-consistent, and world-accurate in the target language.
The translation must sound like original dialogue written in the target language, not a literal translation.

PROJECT CONTEXT:
Show Summary: {show_summary}
Setting / World: {setting}
Genres: {genres}
Linguistic Profile: {linguistic_profile}
Source Language: {source_language}
Target Language: {target_language}

BASE STYLE RULES (always active):
1) Translate meaning, emotional intent, subtext, and character relationships — not literal words.
2) Each character must have a consistent voice across segments. Maintain vocabulary and register.
3) Match genre tone: drama=grounded, action=sharp, comedy=timing-driven, thriller=controlled.
4) Language must match the story world. No anachronistic slang in historical settings.
5) DURATION FITTING (mandatory): fit within the given target_duration seconds.
   - Estimate word budget: target_duration × wps (words per second, adjusted by emotion — see INPUT format)
   - Prioritise emotional core over peripheral content when cutting for time
   - Use commas (brief pause), em-dashes (medium break), ellipses (longer hesitation)
6) Use only {target_language}. No unnecessary code-mixing.
7) When literal conflicts with emotion/lip-sync/genre: choose adaptive.

ACTIVE STYLE TEMPLATE:
{style_template}

ELEVENLABS V3 AUDIO TAGS (mandatory where emotionally appropriate):
Inject 0–3 tags per segment at the START of the phrase they apply to.
Allowed tags: [angry] [sad] [excited] [calm] [scared] [whispers] [shouts] [laughs] [sighs] [gasps] [fast-paced] [drawn out] [sarcastic] [hesitant]
Do not invent tags outside this list.

INPUT FORMAT (JSON array):
[{{
  "ID":            integer — segment index
  "duration":      float   — target spoken duration in seconds
  "word_budget":   integer — target word count (duration × wps for this emotion)
  "dialogue":      string  — original {source_language} text to translate
  "emotion_hint":  string  — audio-feature summary (e.g. "high_energy_fast")
}}]

OUTPUT FORMAT (strict JSON array, no markdown, no commentary):
[{{
  "index":              integer — matches input ID
  "translated_text":    string  — {target_language} translation with inline [audio_tags]
  "roman_script":       string  — romanized phonetic version for dubbing artists
  "estimated_duration": float   — your estimated spoken duration in seconds
  "tags_used":          array   — list of tag strings used
}}]

Rules: preserve order, do not skip or merge lines, output only valid JSON.'''

wps_base = ENGLISH_WPM_DEFAULT / 60.0
SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format(
    show_summary=SHOW_SUMMARY,
    setting=SHOW_SETTING,
    genres=SHOW_GENRES,
    linguistic_profile=LINGUISTIC_PROFILE,
    source_language=SRC,
    target_language=TGT,
    style_template=STYLE_TEMPLATE if STYLE_TEMPLATE else '[Use BASE STYLE RULES above]',
)
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')


In [ ]:
# ── Audio feature extraction per segment ──────────────────────────────────────
# We extract pitch (F0), energy (RMS), and speech rate (ZCR proxy).
# These feed two purposes:
#   1. emotion_hint string fed to the LLM for tag selection
#   2. WPS adjustment: shouted lines -> higher WPM target; whispered -> lower

y_full, sr_full = librosa.load(VOCALS_WAV, sr=None, mono=True)

def classify_emotion_hint(y_seg, sr):
    if len(y_seg) < 512:
        return 'neutral', ENGLISH_WPM_DEFAULT
    try:
        f0, voiced, _ = librosa.pyin(y_seg, fmin=60, fmax=600, frame_length=2048, hop_length=512)
        pitch_mean = float(np.nanmean(f0[voiced])) if voiced is not None and voiced.any() else 0.0
    except Exception:
        pitch_mean = 0.0
    rms = float(np.mean(librosa.feature.rms(y=y_seg)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=y_seg)))

    parts = []
    if rms > 0.08:  parts.append('high_energy')
    elif rms < 0.02: parts.append('low_energy')
    if zcr > 0.15:  parts.append('fast')
    elif zcr < 0.05: parts.append('slow')

    hint = '_'.join(parts) if parts else 'neutral'
    wpm  = ENGLISH_WPM_BY_EMOTION.get(hint, ENGLISH_WPM_DEFAULT)
    return hint, wpm

print('Extracting audio features...')
for seg in tqdm(segments, desc='Audio features'):
    s = int(seg['start'] * sr_full)
    e = int(seg['end']   * sr_full)
    chunk = y_full[s:e]
    seg['emotion_hint'], seg['wpm_target'] = classify_emotion_hint(chunk, sr_full)
    seg['word_budget'] = max(1, round((seg['end']-seg['start']) * seg['wpm_target'] / 60.0))

print('Sample:')
for seg in segments[:4]:
    print(f'  [{seg["start"]:.1f}-{seg["end"]:.1f}s] hint={seg["emotion_hint"]} wpm={seg["wpm_target"]} budget={seg["word_budget"]}w')


## Optional: MOSS-Audio emotion classification (alternative to librosa)

The librosa cell above uses RMS + ZCR heuristics — fast but coarse. MOSS-Audio's Speaker, Emotion & Event Analysis understands emotional state from tone, timbre, and contextual cues — richer signal, especially for speech with mixed emotional content.

Run this cell **after** the librosa cell to compare, or replace the librosa hints by uncommenting the override block at the bottom. Either way, the translation loop uses whatever `seg['emotion_hint']` is set to.

**When to prefer MOSS-Audio emotion over librosa:**
- Segments where librosa labels don't match what you hear (e.g., loud music bleeding into RMS)
- Content with nuanced emotions (fearful, surprised) that RMS/ZCR can't distinguish from high-energy

**Caveats:** ~1-2s per segment on GPU. For a 100-segment video that's 2-3 extra minutes. Tamil/Hindi emotional cues: untested.

In [ ]:
# ── MOSS-Audio per-segment emotion classification ─────────────────────────────
# Run AFTER the librosa audio-features cell. Adds a comparison and optionally
# overrides emotion_hint values used in the translation loop.
MOSS_MODEL_ID_E = 'OpenMOSS-Team/MOSS-Audio-4B-Instruct'
VALID_EMOTIONS  = {'neutral', 'happy', 'sad', 'angry', 'fearful', 'surprised', 'disgusted'}
MOSS_EMO_PROMPT = (
    'What is the emotional state of the speaker in this audio? '
    'Reply with exactly one word from: neutral, happy, sad, angry, fearful, surprised, disgusted.'
)

try:
    import torch
    from transformers import AutoModel, AutoProcessor

    # Reuse model if already loaded in this kernel session
    if 'moss_model_e' not in dir():
        device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Loading {MOSS_MODEL_ID_E}...')
        moss_model_e = AutoModel.from_pretrained(
            MOSS_MODEL_ID_E,
            trust_remote_code=True,
            torch_dtype=(torch.float16 if device == 'mps' else torch.bfloat16 if device == 'cuda' else torch.float32),
            device_map='auto',
        )
        moss_model_e.eval()
        moss_proc_e = AutoProcessor.from_pretrained(
            MOSS_MODEL_ID_E,
            trust_remote_code=True,
        )
    mel_sr_e = getattr(getattr(moss_proc_e, 'config', None), 'mel_sr', 16000)

    if mel_sr_e != sr_full:
        y_moss_e, _ = librosa.load(VOCALS_WAV, sr=mel_sr_e, mono=True)
    else:
        y_moss_e = y_full

    print(f'Classifying {len(segments)} segments with MOSS-Audio emotion...')
    t0 = time.time()
    moss_seg_emotions = {}  # index -> emotion label

    for i, seg in enumerate(tqdm(segments, desc='MOSS emotion')):
        s = int(seg['start'] * mel_sr_e)
        e = int(seg['end']   * mel_sr_e)
        chunk = y_moss_e[s:e]

        if len(chunk) < mel_sr_e * 0.1:
            moss_seg_emotions[i] = 'neutral'
            continue

        inputs = moss_proc_e(text=MOSS_EMO_PROMPT, audios=[chunk], return_tensors='pt')
        inputs = {k: v.to(moss_model_e.device) for k, v in inputs.items()}
        if inputs.get('audio_data') is not None:
            inputs['audio_data'] = inputs['audio_data'].to(moss_model_e.dtype)
        inputs['audio_input_mask'] = inputs['input_ids'] == moss_proc_e.audio_token_id

        with torch.no_grad():
            gen_ids = moss_model_e.generate(
                **inputs, max_new_tokens=16, do_sample=False, use_cache=True,
            )

        raw_emotion = moss_proc_e.decode(
            gen_ids[0, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().lower().split()[0] if gen_ids.shape[1] > inputs['input_ids'].shape[1] else 'neutral'

        moss_seg_emotions[i] = raw_emotion if raw_emotion in VALID_EMOTIONS else 'neutral'

    elapsed = time.time() - t0
    print(f'Done in {elapsed:.1f}s')

    # Compare against librosa hints
    emo_dist = pd.Series(list(moss_seg_emotions.values())).value_counts().to_dict()
    print(f'MOSS-Audio emotion distribution: {emo_dist}')

    agree = sum(
        1 for seg in segments
        if moss_seg_emotions.get(i) == seg.get('emotion_hint', 'neutral').split('_')[0]
    )
    print(f'Librosa vs MOSS-Audio agreement: {agree}/{len(segments)} segments ({100*agree/len(segments):.0f}%)')

    print('\nSample comparison (first 8 segments):')
    for seg in segments[:8]:
        librosa_hint = seg.get('emotion_hint', 'neutral')
        moss_label   = moss_seg_emotions.get(i, '—')
        match        = '✓' if librosa_hint.split('_')[0] == moss_label else '✗'
        print(f'  [{seg["start"]:.1f}-{seg["end"]:.1f}s] librosa={librosa_hint:20s} moss={moss_label:12s} {match}')

    # ── Override emotion hints with MOSS-Audio labels ──────────────────────────
    # Uncomment this block to use MOSS-Audio emotions in the translation loop.
    # Run the librosa audio-features cell first (for wpm_target baseline),
    # then run this to replace emotion_hint only.
    #
    # for seg in segments:
    #     new_emotion = moss_seg_emotions.get(i, seg.get('emotion_hint', 'neutral'))
    #     seg['emotion_hint'] = new_emotion
    #     seg['wpm_target']   = ENGLISH_WPM_BY_EMOTION.get(new_emotion, ENGLISH_WPM_DEFAULT)
    #     seg['word_budget']  = max(1, round((seg['end'] - seg['start']) * seg['wpm_target'] / 60.0))
    # print('Segments updated with MOSS-Audio emotion labels.')

except Exception as e:
    _is_oom = 'memory' in str(e).lower() or 'out of mem' in str(e).lower()
    if _is_oom:
        print('MOSS-Audio SKIPPED: not enough VRAM on this machine (needs ~10 GB).')
        print('Run this cell on GCP with a CUDA GPU — everything else continues fine.')
    else:
        print(f'MOSS-Audio emotion FAILED: {e}')
    try:
        import torch; torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
    except Exception: pass
    moss_seg_emotions = {}  # empty — librosa-based emotion_hint stays in effect

In [ ]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or getpass('Gemini API key: ')
genai.configure(api_key=GEMINI_API_KEY)
print('Gemini ready.')


In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def strip_tags(text):
    '''Remove [audio_tags] before counting words.'''
    return re.sub(r'\[[^\]]+\]', '', text)

def estimate_duration(text, wpm):
    '''Estimate spoken duration from word count and per-emotion WPM.'''
    words = strip_tags(text).split()
    return len(words) / (wpm / 60.0) if words else 0.0

def duration_ok(est, target):
    ratio = est / target if target > 0 else 1.0
    return (1-DURATION_TOLERANCE) <= ratio <= (1+DURATION_TOLERANCE), ratio

def call_gemini(model_name, batch_payload, system_prompt, retry_note=''):
    '''Single Gemini API call. Returns parsed JSON list.'''
    user_msg = f'{retry_note}Translate the following dialogue:\n{json.dumps(batch_payload, ensure_ascii=False, indent=2)}'
    model = genai.GenerativeModel(
        model_name,
        system_instruction=system_prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.35,
            response_mime_type='application/json',
        )
    )
    response = model.generate_content(user_msg)
    raw = response.text.strip()
    if raw.startswith('```'): raw = raw.split('\n', 1)[1].rsplit('```', 1)[0]
    return json.loads(raw)

print('Helpers ready.')


In [ ]:
# ── Core translation function with real duration retry ─────────────────────────
# FIX: previous version only retried on API exceptions.
# This version retries when duration_ok() returns False — up to MAX_DURATION_RETRIES.

BATCH_SIZE = 15   # segments per API call

def translate_all(model_name, label):
    cache_path = os.path.join(TRANSLATION_DIR, f'translation_{label}.json')
    if os.path.exists(cache_path):
        print(f'[{label}] Cached -> {cache_path}')
        with open(cache_path, encoding='utf-8') as f:
            return json.load(f)

    all_translated = []
    t0_total = time.time()

    for batch_start in range(0, len(segments), BATCH_SIZE):
        batch   = segments[batch_start : batch_start + BATCH_SIZE]
        b_idx   = batch_start // BATCH_SIZE + 1
        b_total = (len(segments) + BATCH_SIZE - 1) // BATCH_SIZE

        payload = [
            {'ID': i+1, 'duration': round(s['end']-s['start'], 2),
             'word_budget': s['word_budget'], 'dialogue': s['text'],
             'emotion_hint': s.get('emotion_hint', 'neutral')}
            for i, s in enumerate(batch)
        ]

        print(f'[{label}] Batch {b_idx}/{b_total}  ({len(batch)} segs)  ', end='', flush=True)

        # ── K=3 self-consistency candidates ────────────────────────────────────
        # Generate 3 independent translations, keep the one closest to target duration.
        K_CANDIDATES = 3
        best_results = None

        for k in range(K_CANDIDATES):
            for api_attempt in range(3):  # API exception retries
                try:
                    candidates = call_gemini(model_name, payload, SYSTEM_PROMPT)
                    break
                except Exception as e:
                    print(f'\n  API error attempt {api_attempt+1}: {e}')
                    time.sleep(2 ** api_attempt)
            else:
                candidates = []

            if not candidates:
                continue

            # Score: sum of abs(ratio-1) across all segments (lower = better fit)
            score = 0.0
            for i, seg in enumerate(batch):
                if i >= len(candidates): break
                dur_t = seg['end'] - seg['start']
                text  = candidates[i].get('translated_text', '')
                est   = estimate_duration(text, seg.get('wpm_target', ENGLISH_WPM_DEFAULT))
                ratio = est / dur_t if dur_t > 0 else 1.0
                score += abs(ratio - 1.0)

            if best_results is None or score < best_results[1]:
                best_results = (candidates, score)

        if best_results is None:
            print('FAILED all candidates')
            all_translated.extend([
                {'index': batch_start+i, 'speaker': s['speaker'], 'start': s['start'],
                 'end': s['end'], 'source_text': s['text'], 'translated_text': '',
                 'roman_script': '', 'tags_used': [], 'target_duration': s['end']-s['start'],
                 'estimated_duration': 0.0, 'duration_ratio': 0.0, 'duration_ok': False,
                 'emotion_hint': s.get('emotion_hint', ''), 'retries': 0}
                for i, s in enumerate(batch)
            ])
            continue

        raw_results = best_results[0]

        # ── Per-segment duration retry ──────────────────────────────────────────
        # For segments outside tolerance, re-prompt individually with tighter constraint.
        for i, seg in enumerate(batch):
            if i >= len(raw_results): break
            res      = raw_results[i]
            dur_t    = seg['end'] - seg['start']
            wpm_t    = seg.get('wpm_target', ENGLISH_WPM_DEFAULT)
            text     = res.get('translated_text', '')
            est      = estimate_duration(text, wpm_t)
            ok, ratio = duration_ok(est, dur_t)
            retries   = 0

            while not ok and retries < MAX_DURATION_RETRIES:
                retries += 1
                direction = 'shorter' if ratio > 1+DURATION_TOLERANCE else 'longer'
                target_w  = round(dur_t * wpm_t / 60.0)
                note = (f'RETRY {retries}/{MAX_DURATION_RETRIES}: The previous translation '
                        f'was too {"long" if ratio>1 else "short"}. '
                        f'Previous ratio={ratio:.2f}. '
                        f'Rewrite it to be {direction}. '
                        f'Strict word budget: {target_w} words (±2). ')
                retry_payload = [{'ID': 1, 'duration': round(dur_t, 2),
                                  'word_budget': target_w,
                                  'dialogue': seg['text'],
                                  'emotion_hint': seg.get('emotion_hint', 'neutral')}]
                try:
                    retry_res  = call_gemini(model_name, retry_payload, SYSTEM_PROMPT, retry_note=note)
                    new_text   = retry_res[0].get('translated_text', '') if retry_res else text
                    new_est    = estimate_duration(new_text, wpm_t)
                    new_ok, new_ratio = duration_ok(new_est, dur_t)
                    if new_ok or abs(new_ratio-1) < abs(ratio-1):
                        text = new_text
                        est  = new_est
                        ok   = new_ok
                        ratio = new_ratio
                        res  = retry_res[0]
                except Exception as e:
                    print(f'\n  Retry {retries} failed: {e}')
                time.sleep(0.5)

            all_translated.append({
                'index':             batch_start + i,
                'speaker':           seg['speaker'],
                'start':             seg['start'],
                'end':               seg['end'],
                'source_text':       seg['text'],
                'translated_text':   res.get('translated_text', text),
                'roman_script':      res.get('roman_script', ''),
                'tags_used':         res.get('tags_used', []),
                'target_duration':   round(dur_t, 3),
                'estimated_duration': round(est, 3),
                'duration_ratio':    round(ratio, 3),
                'duration_ok':       ok,
                'emotion_hint':      seg.get('emotion_hint', ''),
                'retries':           retries,
            })

        fit = sum(1 for s in all_translated[-len(batch):] if s['duration_ok'])
        print(f'fit {fit}/{len(batch)}')

    elapsed = time.time() - t0_total
    bad  = sum(1 for s in all_translated if not s['duration_ok'])
    print(f'\n[{label}] Done in {elapsed:.1f}s')
    print(f'  Duration fit: {len(all_translated)-bad}/{len(all_translated)} within ±{int(DURATION_TOLERANCE*100)}%')
    print(f'  Total retries: {sum(s["retries"] for s in all_translated)}')

    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(all_translated, f, indent=2, ensure_ascii=False)
    print(f'  Saved -> {cache_path}')
    return all_translated

print('Translation function ready.')


In [ ]:
results_pro = translate_all(GEMINI_PRO_MODEL, 'gemini_pro')


In [ ]:
results_flash = translate_all(GEMINI_FLASH_MODEL, 'gemini_flash')


## Optional: GPT-4o and Claude Sonnet alternatives

Both use the identical system prompt and `translate_all()` function — they're drop-in replacements for the Gemini models. Enter API keys when prompted, or press Enter to skip and test later. Results are cached so you can run them on-demand without re-running Gemini.

In [ ]:
# ── GPT-4o translation (OpenAI) ───────────────────────────────────────────────
# pip install openai
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key (Enter to skip): ')

results_gpt4o = []
if not OPENAI_API_KEY.strip():
    print('[GPT-4o] Skipped (no API key).')
else:
    try:
        from openai import OpenAI
        oai_client = OpenAI(api_key=OPENAI_API_KEY)

        def call_gpt4o(model_name, batch_payload, system_prompt, retry_note=''):
            user_msg = f'{retry_note}Translate the following dialogue:\n{json.dumps(batch_payload, ensure_ascii=False, indent=2)}'
            resp = oai_client.chat.completions.create(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_msg},
                ],
                temperature=0.35,
                response_format={'type': 'json_object'},
            )
            raw = resp.choices[0].message.content.strip()
            parsed = json.loads(raw)
            # GPT-4o may return {"translations": [...]} — unwrap if needed
            if isinstance(parsed, dict):
                parsed = next((v for v in parsed.values() if isinstance(v, list)), [])
            return parsed

        # Temporarily patch call_gemini to call GPT-4o instead
        _orig = call_gemini
        call_gemini = call_gpt4o
        results_gpt4o = translate_all('gpt-4o', 'gpt4o')
        call_gemini   = _orig
    except ImportError:
        print('openai not installed. Run: pip install openai')
    except Exception as e:
        print(f'GPT-4o FAILED: {e}')

In [ ]:
# ── Claude Sonnet translation (Anthropic) ─────────────────────────────────────
# pip install anthropic
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY') or getpass('Anthropic API key (Enter to skip): ')

results_claude = []
if not ANTHROPIC_API_KEY.strip():
    print('[Claude] Skipped (no API key).')
else:
    try:
        import anthropic
        ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

        def call_claude(model_name, batch_payload, system_prompt, retry_note=''):
            user_msg = f'{retry_note}Translate the following dialogue:\n{json.dumps(batch_payload, ensure_ascii=False, indent=2)}'
            msg = ant_client.messages.create(
                model=model_name,
                max_tokens=4096,
                temperature=0.35,
                system=system_prompt,
                messages=[{'role': 'user', 'content': user_msg}],
            )
            raw = msg.content[0].text.strip()
            if raw.startswith('```'): raw = raw.split('\n', 1)[1].rsplit('```', 1)[0]
            return json.loads(raw)

        _orig = call_gemini
        call_gemini   = call_claude
        results_claude = translate_all('claude-sonnet-4-6', 'claude')
        call_gemini    = _orig
    except ImportError:
        print('anthropic not installed. Run: pip install anthropic')
    except Exception as e:
        print(f'Claude FAILED: {e}')

In [ ]:
def compute_metrics(translated, label):
    if not translated: return None
    ratios = [s['duration_ratio'] for s in translated]
    fit    = [s['duration_ok']    for s in translated]
    tags_n = [len(s['tags_used']) for s in translated]
    retries= [s['retries']        for s in translated]
    return {
        'Model':             label,
        'Segments':          len(translated),
        'Duration fit %':    round(sum(fit)/len(fit)*100, 1),
        'Mean ratio':        round(np.mean(ratios), 3),
        'Ratio std':         round(np.std(ratios),  3),
        'Too short (<0.85)': sum(1 for r in ratios if r < 0.85),
        'Too long  (>1.15)': sum(1 for r in ratios if r > 1.15),
        'Avg tags/seg':      round(np.mean(tags_n), 2),
        'Total retries':     sum(retries),
    }

all_model_results = {
    'Gemini 2.5 Pro':   results_pro,
    'Gemini 2.5 Flash': results_flash,
    'GPT-4o':           results_gpt4o,
    'Claude Sonnet':    results_claude,
}

rows = [compute_metrics(r, label) for label, r in all_model_results.items() if r]
df_m = pd.DataFrame([r for r in rows if r is not None])
df_m

In [ ]:
N = 8
for i in range(min(N, len(results_pro))):
    p  = results_pro[i]
    fl = results_flash[i] if i < len(results_flash) else None
    print(f'\n[{p["start"]:.1f}-{p["end"]:.1f}s] {p["speaker"]}  target={p["target_duration"]:.1f}s  hint={p["emotion_hint"]}')
    print(f'  SRC: {p["source_text"][:90]}')
    ok_p  = '\u2705' if p['duration_ok'] else '\u274c'
    ok_fl = '\u2705' if fl and fl['duration_ok'] else '\u274c'
    print(f'  PRO   {ok_p} [{p["duration_ratio"]:.2f}x r{p["retries"]}]: {p["translated_text"][:100]}')
    if fl:
        print(f'  FLASH {ok_fl} [{fl["duration_ratio"]:.2f}x r{fl["retries"]}]: {fl["translated_text"][:100]}')


In [ ]:
# ── Save canonical translation ─────────────────────────────────────────────────
# Change WINNER to whichever model performed best in the metrics table above.
# Options: 'pro', 'flash', 'gpt4o', 'claude'
WINNER = 'pro'

winner_map = {
    'pro':    results_pro,
    'flash':  results_flash,
    'gpt4o':  results_gpt4o,
    'claude': results_claude,
}
final = winner_map.get(WINNER) or results_pro
if not final:
    raise RuntimeError(f'No results for WINNER={WINNER}. Run the corresponding model cell first.')

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)

fit = sum(1 for s in final if s['duration_ok'])
print(f'Winner: {WINNER.upper()}')
print(f'Saved {len(final)} segments -> {OUT_JSON}')
print(f'Duration fit: {fit}/{len(final)}')